# Entrenamiento de speechBCI en Google Colab

Ejecuta las celdas **en orden**, en un **entorno alojado de Colab con GPU**
(Entorno de ejecución → Cambiar tipo de entorno de ejecución → GPU).
Este notebook llama a `run.py`: la arquitectura, CTC, PER y el entrenamiento
siguen en `src/` y los hiperparámetros en `configs/speech_original.yaml`.

Se mantienen las 19 sesiones, 6.640 frases de train, 680 de test, cinco GRU
de 1.024 unidades, 10.001 actualizaciones y lote efectivo de 64 en grupos de 4.
**Esta es la versión completa; su duración no está medida ni se garantiza
que cumpla el límite docente de 20 minutos.**

Solo necesitas este `.ipynb`, una cuenta de W&B y autorizar Google Drive.
Incluye un snapshot verificable del código: no necesita clonar el repositorio
ni subir los datos desde el PC. Descarga los datos públicos de Dryad.
La primera celda impide usarlo accidentalmente fuera de Colab.


In [ ]:
#@title 1. Comprobar que estamos en Colab con GPU
from pathlib import Path
import os
import subprocess
import sys

try:
    import google.colab
except ImportError as error:
    raise RuntimeError("Abre este notebook en Google Colab; no se ejecuta en el PC local.") from error

COLAB_BASE = globals().get("COLAB_BASE", Path.cwd().resolve())
PROJECT = COLAB_BASE / "tic_proyecto_colab"
try:
    subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,driver_version", "--format=csv"], check=True)
except (FileNotFoundError, subprocess.CalledProcessError) as error:
    raise RuntimeError("Selecciona una GPU en Entorno de ejecución → Cambiar tipo de entorno de ejecución y vuelve a empezar.") from error
print("La instalación, preparación y entrenamiento se ejecutarán en la máquina de Colab.")


## Código de la entrega

La celda siguiente contiene un ZIP pequeño de las fuentes y su SHA-256.
Comprueba también cada archivo extraído. No contiene señales neuronales,
pesos, credenciales ni el entorno del PC. El commit base y los hashes quedan
registrados con el experimento. Para actualizar este snapshot desde el
repositorio: `python scripts/export_colab.py`.


In [ ]:
#@title 2. Copiar el código del proyecto (snapshot incluido)
import base64
import hashlib
import io
import json
from pathlib import PurePosixPath
from zipfile import ZipFile

SOURCE_SHA256 = '35c74840e43090b4f80b013f727ce2a2e9decfc7778ed4788706f262b5ced187'
SOURCE_B64 = (
    '...'
    'Lmpzb25QSwUGAAAAAB4AHgD8BwAA3ZsAAAAA'
)

SOURCE_BYTES = base64.b64decode(SOURCE_B64)
if hashlib.sha256(SOURCE_BYTES).hexdigest() != SOURCE_SHA256:
    raise ValueError("El paquete de código no supera su SHA-256.")
PROJECT.mkdir(parents=True, exist_ok=True)
with ZipFile(io.BytesIO(SOURCE_BYTES)) as archive:
    for member in archive.infolist():
        relative = PurePosixPath(member.filename)
        if relative.is_absolute() or ".." in relative.parts or member.is_dir():
            raise ValueError(f"Ruta inesperada en el paquete: {member.filename}")
        destination = PROJECT.joinpath(*relative.parts)
        destination.parent.mkdir(parents=True, exist_ok=True)
        destination.write_bytes(archive.read(member))
SOURCE_MANIFEST = json.loads((PROJECT / "source_snapshot.json").read_text())
for name, expected in SOURCE_MANIFEST["files"].items():
    if hashlib.sha256((PROJECT / name).read_bytes()).hexdigest() != expected:
        raise ValueError(f"Código distinto del snapshot: {name}")
print("Código preparado:", len(SOURCE_MANIFEST["files"]), "archivos")
print("Commit base:", SOURCE_MANIFEST["git_commit_base"])
print("SHA-256 del paquete:", SOURCE_SHA256)


## Entorno aislado

Se crea un Python 3.12.11 independiente del kernel de Colab, usando uv 0.9.4.
Se instalan PyTorch 2.5.1 / torchvision 0.20.1 con CUDA 12.4 y las versiones
del `requirements.txt`. Así no se sustituyen NumPy ni las librerías del kernel
de Colab. La descarga inicial de dependencias puede tardar varios minutos.

Referencias: [instalación de Python con uv](https://docs.astral.sh/uv/guides/install-python/)
y [versiones anteriores de PyTorch](https://pytorch.org/get-started/previous-versions/).


In [ ]:
#@title 3. Instalar el entorno del proyecto
import shutil

subprocess.run([sys.executable, "-m", "pip", "install", "--disable-pip-version-check", "uv==0.9.4"], check=True)
UV = shutil.which("uv")
if UV is None:
    raise RuntimeError("No se encontró uv después de instalarlo.")
PYTHON = PROJECT / ".venv" / "bin" / "python"
if not PYTHON.is_file():
    subprocess.run([UV, "venv", "--python", "3.12.11", "--seed", str(PROJECT / ".venv")], check=True)
subprocess.run([UV, "pip", "install", "--python", str(PYTHON), "--index-url", "https://download.pytorch.org/whl/cu124",
                "torch==2.5.1", "torchvision==0.20.1"], cwd=PROJECT, check=True)
subprocess.run([UV, "pip", "install", "--python", str(PYTHON), "-r", "requirements.txt"], cwd=PROJECT, check=True)
subprocess.run([UV, "pip", "check", "--python", str(PYTHON)], cwd=PROJECT, check=True)

def ejecutar(argumentos: list[str]) -> None:
    """Ejecuta un comando del proyecto y detiene la celda si falla."""
    subprocess.run([str(PYTHON), *argumentos], cwd=PROJECT, check=True)

print("Entorno aislado preparado. No hace falta reiniciar el kernel.")


## Guardar los resultados en Drive

Autoriza el montaje de Drive. El proyecto, las señales y los paquetes se
mantienen en el disco temporal de Colab para entrenar; solo los checkpoints,
la configuración y los registros se guardan en la carpeta indicada de Drive.
La cuenta de W&B registrará las métricas online.

Colab puede terminar una sesión y no garantiza una GPU ni duración concretas.
Se guarda un checkpoint tras las evaluaciones, aproximadamente cada 50 pasos.
**Los checkpoints actuales permiten evaluar e inferir; todavía no permiten
reanudar exactamente el optimizador.** Si se pierde la sesión, el modelo
guardado se conserva para evaluarlo, pero volver a entrenar empieza desde cero.
[Información oficial de Colab](https://research.google.com/colaboratory/faq.html).


In [ ]:
#@title 4. Conectar Drive y preparar la configuración
CARPETA_DRIVE = "speech_bci" #@param {type:"string"}

from datetime import datetime
from zoneinfo import ZoneInfo
from uuid import uuid4
from google.colab import drive
import json

relative = Path(CARPETA_DRIVE)
if not CARPETA_DRIVE.strip() or relative.is_absolute() or ".." in relative.parts:
    raise ValueError("Indica una carpeta relativa a Mi unidad, por ejemplo speech_bci.")
DRIVE_MOUNT = COLAB_BASE / "drive"
drive.mount(str(DRIVE_MOUNT))
DRIVE_ROOT = DRIVE_MOUNT / "MyDrive" / relative
EXPERIMENT_ID = datetime.now(ZoneInfo("Europe/Madrid")).strftime("%Y%m%d_%H%M%S") + "_" + uuid4().hex[:8]
RESULTS = DRIVE_ROOT / "experimentos" / EXPERIMENT_ID
RESULTS.mkdir(parents=True, exist_ok=False)
CHECKPOINTS = DRIVE_ROOT / "checkpoints"
CHECKPOINTS.mkdir(parents=True, exist_ok=True)
link = PROJECT / "checkpoints"
if link.is_symlink():
    if link.resolve() != CHECKPOINTS.resolve():
        raise RuntimeError("El enlace de checkpoints apunta a otra carpeta. Usa el mismo CARPETA_DRIVE o una sesión nueva.")
elif link.exists():
    raise RuntimeError("Existe una carpeta local checkpoints. Revísala antes de crear el enlace a Drive.")
else:
    link.symlink_to(CHECKPOINTS, target_is_directory=True)

CONFIG = "configs/speech_colab.yaml"
config_code = """
from pathlib import Path
import json
import yaml
cfg = yaml.safe_load(Path('configs/speech_original.yaml').read_text())
cfg['experimento']['nombre'] = 'speech_original_colab'
cfg['ejecucion']['dispositivo'] = 'cuda'
Path('configs/speech_colab.yaml').write_text(yaml.safe_dump(cfg, sort_keys=False), encoding='utf-8')
print(json.dumps(cfg))
"""
cfg = json.loads(subprocess.check_output([str(PYTHON), "-c", config_code], cwd=PROJECT, text=True))
for variable in ("OMP_NUM_THREADS", "MKL_NUM_THREADS", "OPENBLAS_NUM_THREADS"):
    os.environ[variable] = str(cfg["ejecucion"]["hilos_cpu"])
(RESULTS / "source_snapshot.json").write_text(json.dumps(SOURCE_MANIFEST, indent=2))
(RESULTS / "fuentes.zip").write_bytes(SOURCE_BYTES)
shutil.copyfile(PROJECT / CONFIG, RESULTS / "config.yaml")
print("Resultados:", RESULTS)
print("Checkpoints:", CHECKPOINTS)
print("Modelo:", cfg["modelo"])
print("Lote efectivo:", cfg["datos"]["batch_size"], "| microlote:", cfg["entrenamiento"]["micro_batch_size"])
print("Actualizaciones:", cfg["entrenamiento"]["pasos"], "| W&B:", cfg["wandb"]["modo"])


In [ ]:
#@title 5. Comprobar CUDA y registrar el entorno
probe = """
import json
import platform
import sys
import torch
import numpy
import wandb
if not torch.cuda.is_available():
    raise RuntimeError('El Python del proyecto no detecta CUDA. Comprueba que Colab tiene GPU.')
major, minor = torch.cuda.get_device_capability()
maximum = max(int(name.removeprefix('sm_')) // 10 for name in torch.cuda.get_arch_list() if name.startswith('sm_'))
if major > maximum:
    raise RuntimeError('La GPU asignada necesita un PyTorch más reciente. Para conservar torch 2.5.1 usa una T4, L4 o A100 si están disponibles.')
print(json.dumps({'python': sys.version, 'platform': platform.platform(), 'torch': str(torch.__version__),
                  'numpy': numpy.__version__, 'wandb': wandb.__version__, 'cuda': torch.version.cuda,
                  'gpu': torch.cuda.get_device_name(), 'capability': [major, minor],
                  'gpu_total_mib': torch.cuda.get_device_properties(0).total_memory / (1 << 20)}, indent=2))
"""
environment = json.loads(subprocess.check_output([str(PYTHON), "-c", probe], cwd=PROJECT, text=True))
(RESULTS / "entorno.json").write_text(json.dumps(environment, indent=2))
freeze = subprocess.check_output([str(PYTHON), "-m", "pip", "freeze"], cwd=PROJECT, text=True)
(RESULTS / "paquetes.txt").write_text(freeze)
print(json.dumps(environment, indent=2))
print("Espacio libre en Colab:", round(shutil.disk_usage(PROJECT).free / (1 << 30), 1), "GiB")


## Descargar y preparar los mismos datos

Se descarga `competitionData.tar.gz` (**3,67 GB**) desde Dryad y se verifica
el tamaño y SHA-256 oficial antes de extraer. El archivo completo ocupa unos
**6,95 GB extraído** y la caché seleccionada añade **2,44 GB**. No se usa holdout
para entrenar o medir PER. La preparación ejecuta los scripts del repositorio:
fonemas con g2p-en, estadísticas solo de train y señales NPY por sesión.

Si Dryad rechaza la descarga automática, sube el `.tar.gz` que ya tienes a
Drive e indica su ruta **relativa a Mi unidad** en la celda siguiente, por
ejemplo `speech_bci/datos/competitionData.tar.gz`. También se comprobará su
SHA-256. No subas el archivo grande mediante el formulario del notebook.


In [ ]:
#@title 6. Obtener y extraer competitionData
ARCHIVO_DATOS_DRIVE = "" #@param {type:"string"}

destination = PROJECT / "data" / "speech_2023" / "competitionData.tar.gz"
if ARCHIVO_DATOS_DRIVE.strip():
    relative = Path(ARCHIVO_DATOS_DRIVE.strip())
    if relative.is_absolute() or ".." in relative.parts:
        raise ValueError("La ruta debe ser relativa a Mi unidad.")
    source = DRIVE_MOUNT / "MyDrive" / relative
    if not source.is_file():
        raise FileNotFoundError(source)
    destination.parent.mkdir(parents=True, exist_ok=True)
    if not destination.exists():
        shutil.copyfile(source, destination)
    ejecutar(["scripts/get_data.py", "--dataset", "speech_2023", "--files", "competitionData.tar.gz", "--verify-only"])
else:
    ejecutar(["scripts/get_data.py", "--dataset", "speech_2023", "--files", "competitionData.tar.gz"])

extract = """
from pathlib import Path
import json
import tarfile
import yaml
cfg = yaml.safe_load(Path('configs/speech_colab.yaml').read_text())
manifest = json.loads(Path('docs/datasets/speech_2023/manifest.json').read_text())
digest = next(item['sha256'] for item in manifest['files'] if item['name'] == 'competitionData.tar.gz')
marker = Path('data/speech_2023/.extraccion_verificada')
expected = [Path(cfg['datos']['raiz']) / split / (session + '.mat') for split in ['train', 'test'] for session in cfg['datos']['sesiones']]
if not marker.exists() or marker.read_text().strip() != digest or not all(path.is_file() for path in expected):
    with tarfile.open('data/speech_2023/competitionData.tar.gz') as archive:
        archive.extractall('data/speech_2023', filter='data')
    marker.write_text(digest)
print('Extraídas las señales originales. Las 19 sesiones seleccionadas están disponibles.')
"""
ejecutar(["-c", extract])


In [ ]:
#@title 7. Preparar recursos, fonemas y señales
REHACER_PREPARACION = False #@param {type:"boolean"}

ejecutar(["scripts/get_phoneme_resources.py", "--config", CONFIG])
labels = PROJECT / cfg["etiquetas"]["salida"] / "manifest.json"
signals = PROJECT / cfg["datos"]["preparados"] / "manifest.json"
if REHACER_PREPARACION or not labels.is_file():
    ejecutar(["scripts/prepare_phoneme_labels.py", "--config", CONFIG])
if REHACER_PREPARACION or not signals.is_file():
    ejecutar(["scripts/prepare_speech_data.py", "--config", CONFIG])
verify = """
from pathlib import Path
import json
import yaml
from src.data.speech_dataset import SpeechDataset
cfg = yaml.safe_load(Path('configs/speech_colab.yaml').read_text())
train = SpeechDataset(cfg, 'train')
test = SpeechDataset(cfg, 'test')
if (len(train), len(test), len(train.sessions)) != (6640, 680, 19):
    raise ValueError('Los datos no corresponden al baseline de 19 sesiones.')
print(json.dumps({'train': len(train), 'test': len(test), 'sesiones': len(train.sessions), 'normalizacion': 'solo train', 'hashes': 'verificados'}, indent=2))
"""
ejecutar(["-c", verify])


## Autenticar W&B

En **Secretos** de Colab (icono de llave), crea `WANDB_API_KEY` y permite el
acceso de este notebook. Puedes obtener la clave en
[wandb.ai/authorize](https://wandb.ai/authorize). Si no configuras el secreto,
la siguiente celda la pide mediante una entrada oculta. No pegues la clave
como texto dentro del código ni en una salida del notebook.

El proyecto de W&B será `speech-bci` en la cuenta de esa clave. El código
exige W&B online u offline; aquí se conserva **online**.


In [ ]:
#@title 8. Cargar la credencial de W&B sin mostrarla
import getpass
from google.colab import userdata

try:
    api_key = userdata.get("WANDB_API_KEY")
except (userdata.SecretNotFoundError, userdata.NotebookAccessError, userdata.TimeoutException):
    api_key = getpass.getpass("API key de W&B (entrada oculta): ")
if not api_key or not api_key.strip():
    raise ValueError("Falta la credencial de W&B.")
os.environ["WANDB_API_KEY"] = api_key.strip()
del api_key
print("Credencial cargada. W&B mostrará el enlace del run al iniciar el entrenamiento.")


## Entrenamiento completo

La siguiente celda inicia el entrenamiento **desde cero** con
`python run.py --config configs/speech_colab.yaml`. Mantén la celda en ejecución.
Mostrará pérdidas, learning rate, memoria y PER del **test publicado**.
PER se guarda como proporción e incluye SIL; no es WER ni accuracy.

El código no cambia el tamaño del modelo según la GPU. Los microlotes de 4
reducen la memoria sin cambiar las 64 frases de cada actualización. Si se
interrumpe la celda se intenta parar el proceso de entrenamiento; no se deja
un entrenamiento en segundo plano. La última evaluación guardada puede
consultarse en Drive, aunque no haya terminado el run.


In [ ]:
#@title 9. Entrenar y guardar la salida en Drive
import signal
import time

if not os.environ.get("WANDB_API_KEY"):
    raise RuntimeError("Ejecuta primero la celda de autenticación de W&B.")
shutil.copyfile(PROJECT / CONFIG, RESULTS / "config.yaml")
before = set((CHECKPOINTS / "speech_original").glob("*/ultimo.pt"))
started = time.perf_counter()
process = None
status = {"completo": False, "estado": "iniciado", "config": CONFIG}
LAST_CHECKPOINT = None
try:
    with (RESULTS / "entrenamiento.log").open("w", encoding="utf-8") as log:
        process = subprocess.Popen([str(PYTHON), "-u", "run.py", "--config", CONFIG], cwd=PROJECT,
                                   stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                                   text=True, encoding="utf-8", errors="replace", bufsize=1)
        for line in process.stdout:
            print(line, end="", flush=True)
            log.write(line)
            log.flush()
        returncode = process.wait()
    status.update(estado="terminado" if returncode == 0 else "fallido", returncode=returncode, completo=returncode == 0)
    if returncode:
        raise RuntimeError(f"El entrenamiento terminó con código {returncode}; revisa la salida y el log en Drive.")
except KeyboardInterrupt:
    status["estado"] = "interrumpido"
    raise
except BaseException:
    status["estado"] = "fallido"
    raise
finally:
    if process is not None and process.poll() is None:
        process.send_signal(signal.SIGINT)
        try:
            process.wait(timeout=30)
        except subprocess.TimeoutExpired:
            process.terminate()
            try:
                process.wait(timeout=10)
            except subprocess.TimeoutExpired:
                process.kill()
                process.wait()
    created = set((CHECKPOINTS / "speech_original").glob("*/ultimo.pt")) - before
    if created:
        LAST_CHECKPOINT = max(created, key=lambda path: path.stat().st_mtime)
        status["checkpoint"] = str(LAST_CHECKPOINT)
    status["segundos_pared"] = time.perf_counter() - started
    if process is not None:
        status["returncode"] = process.returncode
    (RESULTS / "estado.json").write_text(json.dumps(status, indent=2), encoding="utf-8")
    print("\nRegistros en Drive:", RESULTS)
    print("Checkpoint:", LAST_CHECKPOINT or "Todavía no se había guardado ningún checkpoint de este run.")


## Consultar los archivos guardados

Puedes ejecutar esta celda tras una interrupción o al terminar. Selecciona
un checkpoint ya existente indicando su ruta relativa a la carpeta de
checkpoints de Drive, por ejemplo `speech_original/ID_RUN/ultimo.pt`.
Si queda vacío, usa el guardado por la celda de entrenamiento actual.
`ultimo.json` contiene las métricas y el paso realmente completado.

No copies la cifra de una comprobación corta como resultado de la réplica
completa. La interpretación y comparación de los resultados corresponden
al autor del trabajo.


In [ ]:
#@title 10. Mostrar los resultados guardados (sin entrenar)
CHECKPOINT_GUARDADO = "" #@param {type:"string"}

selected = globals().get("LAST_CHECKPOINT")
if CHECKPOINT_GUARDADO.strip():
    relative = Path(CHECKPOINT_GUARDADO.strip())
    if relative.is_absolute() or ".." in relative.parts:
        raise ValueError("La ruta debe ser relativa a la carpeta de checkpoints de Drive.")
    selected = CHECKPOINTS / relative
if selected is None:
    print("Carpetas disponibles:")
    for path in sorted((CHECKPOINTS / "speech_original").glob("*/ultimo.json")):
        print(path.relative_to(CHECKPOINTS))
    print("Indica CHECKPOINT_GUARDADO para ver uno de los resultados anteriores.")
else:
    if not selected.is_file():
        raise FileNotFoundError(selected)
    summary = selected.with_suffix(".json")
    print("Pesos:", selected)
    print(json.dumps(json.loads(summary.read_text()), indent=2))
print("Config, entorno, código, log y estado de esta sesión:", RESULTS)


Para volver a evaluar los pesos en este entorno, usa el mismo `run.py`:

```python
ejecutar(["run.py", "--config", CONFIG, "--solo-evaluar", "--checkpoint", str(selected)])
```

Se necesita mantener los datos preparados y haber cargado la credencial de W&B.
Esta llamada evalúa las 680 frases y abre un run de evaluación; no reanuda el
entrenamiento. Para una sesión de Colab nueva, ejecuta primero las celdas 1–8.
